# Stage 3B.02 — freeze manifest, pairing, and reused controls

In [ ]:
import csv,os,subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; P=Path.home()/"LIBERO-plus"; ID=Path.home()/"venv-stage1-id/bin/python"; OOD=Path.home()/"venv-stage1-ood/bin/python"; N=Path.home()/"stage1-native"; S3=Path.home()/"stage3"; OUT=Path.home()/"stage3b"
MAN=OUT/"stage3b_object_layout_manifest.csv"; AUD=OUT/"stage3b_initialization_pairing_audit.csv"; REUSE=OUT/"stage3b_reused_stage3_controls_audit.csv"
bench=subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(); plus=subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
subprocess.run([str(ID),"-m","async_vla_benchmark.scripts.make_stage3b_manifest","--output",str(MAN),"--git-sha",bench,"--lerobot-git-sha","2aba372b4e217cc47db28e0f836859b20d1456c9","--libero-plus-git-sha",plus,"--model-revision","8e174154ef5f6c60a8da12ae99c303d8963138c1"],cwd=R,check=True)
base=os.environ.copy(); base.update({"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
subprocess.run([str(ID),"-m","async_vla_benchmark.scripts.resolve_stage3_initializations","--config",str(R/"async_vla_benchmark/configs/stage3b.yaml"),"--manifest",str(MAN),"--scene","id","--expected-rows","144","--audit-output",str(AUD)],cwd=R,env=base,check=True)
ood=base.copy(); ood.update({"PYTHONPATH":str(P),"MAGICK_HOME":str(N),"PATH":str(N/"bin")+os.pathsep+ood.get("PATH",""),"LD_LIBRARY_PATH":str(N/"lib")+os.pathsep+ood.get("LD_LIBRARY_PATH","")})
subprocess.run([str(OOD),"-m","async_vla_benchmark.scripts.resolve_stage3_initializations","--config",str(R/"async_vla_benchmark/configs/stage3b.yaml"),"--manifest",str(MAN),"--scene","ood","--expected-rows","144","--audit-output",str(AUD)],cwd=R,env=ood,check=True)
subprocess.run([str(ID),"-m","async_vla_benchmark.scripts.audit_stage3b_reused_controls","--stage3-manifest",str(S3/"stage3_manifest.csv"),"--stage3-results",str(S3/"stage3_episode_results.csv"),"--output",str(REUSE)],cwd=R,check=True)
subprocess.run([str(ID),"-m","async_vla_benchmark.scripts.validate_stage3b","--manifest",str(MAN),"--output-dir",str(OUT),"--reuse-audit",str(REUSE),"--allow-incomplete"],cwd=R,check=True)
print("STOP HERE: confirm manifest=144, pairing audit=24, reused goal ID=48")